# Lab 1: Vector Databases & Semantic Indexing

**Session 2 · Lab 1 of 3 · ~35 minutes**

## Goal

Use the pre-built Agent Bricks Knowledge Assistant to perform semantic search over the
financial document corpus. Understand how index design, filtering, and hybrid search
affect the quality of retrieval.

## What you'll do

1. Explore the Knowledge Assistant configuration in the Agent Bricks UI
2. Ask financial questions and observe how the KA retrieves and cites documents
3. Examine the underlying Vector Search index
4. Test company-level filtering
5. Compare ANN (semantic) vs HYBRID (semantic + keyword) search

---

In [0]:
%run ../utils/config

In [0]:
# Knowledge Assistant agent name (as shown in Agent Bricks UI)
KA_AGENT_NAME    = "financial-intelligence-assistant"
VS_INDEX_NAME    = f"{catalog}.{shared_schema}.financial_docs_for_search_index"
VS_ENDPOINT_NAME = "financial-docs-vs-endpoint"        # Vector Search endpoint

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()


# Resolve agent name → serving endpoint via the tiles API
_tiles = w.api_client.do("GET", "/api/2.0/tiles").get("tiles", [])
_match = next((t for t in _tiles if t["name"] == KA_AGENT_NAME), None)

if _match is None:
    raise ValueError(
        f"Agent '{KA_AGENT_NAME}' not found. "
        f"Available agents: {[t['name'] for t in _tiles if t.get('tile_type') == 'KA']}"
    )

KA_ENDPOINT_NAME = _match["serving_endpoint_name"]
print(f"Agent '{KA_AGENT_NAME}' → endpoint '{KA_ENDPOINT_NAME}'")


## Part 1: Explore the Knowledge Assistant

**In the Databricks UI:**
1. Navigate to **Agents** in the Left Nav
2. Open `financial-intelligence-assistant`
3. Review the **Settings** configuration:
   - Optionally provide instructions on how the KA should behave.
   - Descriptions are used by Supervisor Agents to understand what a KA agent can do.
3. Review the **Sources** configuration:
   - **Vector Search index:** `financial_docs_for_search_index`
5. Click `financial_docs_for_search_index` to open the vector search index table in another tab.
   - Observe that there are 125 rows in the index.  A pipeline has been automatically configured to sync the index from a regular delta table.
   - Note that the index table is not a regular Delta table as indicated by the table type `Foreign` and Data source `Vector Index`.  It can therefore not be queried with a regular SQL statement nor on the Sample Data tab.

## Part 2: Ask financial questions

Let's query the Knowledge Assistant some questions.

### Use the Agent UI

In the Databricks UI:
1. Return to **Agents** -> `financial-intelligence-assistant`
2. In the input box at the bottom, enter a question that shoudl be answerable from the documents, for example, try these questions:
- Question 1: `What is NVIDIA's revenue trajectory over the past 3 years? Include specific figures.`
- Question 2: `Which of the companies is investing most heavily in AI infrastructure? What evidence supports this?`

### Use the Databricks SDK

The following cells ask the same questions of the Knowledge Assistant programmatically, using the Databricks Python SDK.

In [0]:
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()

def ask_assistant(question, print_citations=True):
    """Query the financial Knowledge Assistant and print the response."""
    # Agent endpoints use the Responses API format
    payload = {"input": [{"role": "user", "content": question}]}
    resp = w.api_client.do(
        "POST",
        f"/serving-endpoints/{KA_ENDPOINT_NAME}/invocations",
        body=payload,
    )
    # Parse Responses API output
    message = resp["output"][0]
    answer = ""
    annotations = []
    for chunk in message["content"]:
        answer = answer + chunk["text"]
        annotations.extend(chunk.get("annotations", []))
    print(f"Q: {question}")
    print(f"\nA: {answer}")

    # Print source citations if available
    if print_citations:
        if annotations:
            print("\nSources:")
            for ann in annotations:
                displayHTML(f'<a href="{ann['url']}">{ann['title']}</a>')
        else:
            print("  No citations found.")
    return answer

In [0]:
# Question 1: Revenue trajectory — tests retrieval across multiple time periods
ask_assistant("What is NVIDIA's revenue trajectory over the past 3 years? Include specific figures.")

In [0]:
# Question 2: Cross-company comparison — tests breadth of retrieval
ask_assistant("Which of the companies is investing most heavily in AI infrastructure? What evidence supports this?")

In [0]:
# Question 3: Risk analysis — tests nuanced retrieval
ask_assistant("What supply chain and geopolitical risks did these companies highlight in their most recent earnings calls filings?")

**Discuss:** How do the answers compare to what you'd get from a general-purpose LLM without retrieval?
What happens when you ask about very recent earnings (Q3 2025)?

## Part 3: Examine the Vector Search index

Next, we will look more closely at the vector search index underpinning the Knowledge Assistant.

In [0]:
# Inspect the vector search index details

# List available vector search indexes
vs_client = w.vector_search_indexes
indexes = list(vs_client.list_indexes(endpoint_name=VS_ENDPOINT_NAME))

print("Vector Search indexes:")
for mini_idx in indexes:
    idx = vs_client.get_index(index_name=mini_idx.name)
    print(f"  {idx.name}")
    print(f"    Status:  {idx.status.ready}")
    print(f"    Indexed: {idx.status.indexed_row_count:,} chunks")

### Query the vector search index
Include metadata values in the response

In [0]:
# Direct vector search query — returns the raw retrieval results with scores
results = w.vector_search_indexes.query_index(
    index_name=VS_INDEX_NAME,
    query_text="NVIDIA data center revenue growth",
    columns=["company", "fiscal_period", "document_type", "plain_text"],
    num_results=5,
)

print("Top 5 semantic search results for 'NVIDIA data center revenue growth':")
for i, row in enumerate(results.result.data_array or []):
    print(f"\n[{i+1}] Score: {row[-1]:.4f}")
    print(f"    Company: {row[0]}  Period: {row[1]}  Type: {row[2]}")
    print(f"    Text:    {str(row[3])[:500]}...")

**Index design insight:** The Knowledge Assistant automatically chunks `plain_text` into ~1000-token
segments with overlap. This balances:
- **Chunk too small:** Loses context (a financial table split across chunks)
- **Chunk too large:** Dilutes the embedding (one embedding for 10 topics)
- **Overlap:** Ensures boundary sentences aren't lost

## Part 4: Test company-level filtering

Metadata filters allow precise scoping — dramatically improving precision for targeted queries.

In [0]:
# Search WITHOUT filter — returns results from any company
unfiltered = w.vector_search_indexes.query_index(
    index_name=VS_INDEX_NAME,
    query_text="AI datacenter constraints",
    columns=["company", "fiscal_period", "document_type"],
    num_results=5,
)

print("WITHOUT filter — companies returned:")
print(f"  {'Company':<30} {'Period':<25} {'Type'}")
print(f"  {'-'*30} {'-'*25} {'-'*20}")
for row in unfiltered.result.data_array or []:
    print(f"  {str(row[0])[:30]:<30} {str(row[1])[:25]:<25} {row[2]}")

In [0]:
# Search WITH company filter — restrict to NVIDIA only
filtered = w.vector_search_indexes.query_index(
    index_name=VS_INDEX_NAME,
    query_text="AI datacenter constraints",
    columns=["company", "fiscal_period", "document_type"],
    filters_json='{"company": "NVIDIA"}',
    num_results=5,
)

print("WITH company='NVIDIA' filter:")
print(f"  {'Company':<30} {'Period':<25} {'Type'}")
print(f"  {'-'*30} {'-'*25} {'-'*20}")
for row in filtered.result.data_array or []:
    print(f"  {str(row[0])[:30]:<30} {str(row[1])[:25]:<25} {row[2]}")

In [0]:
# Filter by document type — restrict to earnings call transcripts
transcript_results = w.vector_search_indexes.query_index(
    index_name=VS_INDEX_NAME,
    query_text="management outlook for next quarter",
    columns=["company", "fiscal_period", "document_type"],
    filters_json='{"document_type LIKE": "Earnings"}',
    num_results=10,
)

print("Earnings Call transcripts mentioning management outlook:")
print(f"  {'Company':<30} {'Period':<25} {'Type'}")
print(f"  {'-'*30} {'-'*25} {'-'*20}")
for row in transcript_results.result.data_array or []:
    print(f"  {str(row[0])[:30]:<30} {str(row[1])[:25]:<25} {row[2]}")

## Part 5: Compare ANN vs Hybrid search

**ANN (Approximate Nearest Neighbor):** Pure semantic search — finds chunks whose *meaning* is similar

**HYBRID:** Combines semantic similarity + keyword frequency (BM25) — better for queries with specific terms

After the following cell completes, use the table value search to look for "H100".  Note the higher incidents of matches using Hybrid versus ANN.

In [0]:
import pandas as pd

query = "H100 GPU shortage allocation"
result_columns = ["company", "fiscal_period", "plain_text", "score"]

# ANN (pure semantic)
ann_results = w.vector_search_indexes.query_index(
    index_name=VS_INDEX_NAME,
    query_text=query,
    columns=["company", "fiscal_period", "plain_text"],
    num_results=5,
    query_type="ANN",
)

# HYBRID (semantic + keyword)
hybrid_results = w.vector_search_indexes.query_index(
    index_name=VS_INDEX_NAME,
    query_text=query,
    columns=["company", "fiscal_period", "plain_text"],
    num_results=5,
    query_type="HYBRID",
)

print(f"Query: '{query}'\n")

print("\nANN results (table):")
display(pd.DataFrame(ann_results.result.data_array or [], columns=result_columns))
print("\nHYBRID results (table):")
display(pd.DataFrame(hybrid_results.result.data_array or [], columns=result_columns))

**Key insight:** Hybrid search improves precision for product-name queries like "H100" or "Copilot"
where the exact term is important — pure semantic search might return results about "AI hardware"
that never mention the specific product.

---

## Key Takeaways

1. **Agent Bricks Knowledge Assistant** makes building a production RAG system a configuration task, not a coding task
2. **Metadata filters** are the most cost-effective way to improve retrieval precision — always design your schema with filter dimensions in mind
3. **Hybrid search** outperforms pure semantic search for queries with specific proper nouns or technical terms
4. **Index design matters:** Chunk size, overlap, and embedding model choice all affect retrieval quality

---

**Next:** Lab 2 — Knowledge graphs for RAG & agents

→ Open `03_knowledge_graphs`